# Logistic Regression - Manual Implementation

In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv('train.csv')
data.head()

,id,day,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,winddirection,windspeed,rainfall
0,0,1,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,60.0,17.2,1
1,1,2,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,50.0,21.9,1
2,2,3,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,70.0,18.1,1
3,3,4,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,60.0,35.6,1
4,4,5,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,40.0,24.8,0


In [3]:
features = ['pressure', 'maxtemp', 'temparature', 'mintemp', 'dewpoint', 'humidity', 'cloud', 'sunshine','winddirection', 'windspeed']
target = 'rainfall'

In [4]:
# Fill missing values for features
for feature in features:
    data[feature] = data[feature].fillna(data[feature].mean())

In [5]:
X_raw = data[features].values
X_raw

array([[1.0174e+03, 2.1200e+01, 2.0600e+01, ..., 1.1000e+00, 6.0000e+01,
        1.7200e+01],
       [1.0195e+03, 1.6200e+01, 1.6900e+01, ..., 0.0000e+00, 5.0000e+01,
        2.1900e+01],
       [1.0241e+03, 1.9400e+01, 1.6100e+01, ..., 8.3000e+00, 7.0000e+01,
        1.8100e+01],
       ...,
       [1.0133e+03, 1.9000e+01, 1.6300e+01, ..., 5.0000e+00, 4.0000e+01,
        3.2900e+01],
       [1.0223e+03, 1.6400e+01, 1.5200e+01, ..., 1.0000e-01, 4.0000e+01,
        1.8000e+01],
       [1.0138e+03, 2.1200e+01, 1.9100e+01, ..., 1.0000e+00, 7.0000e+01,
        4.8000e+01]], shape=(2190, 10))

In [6]:
Y = data[target].values
Y

array([1, 1, 1, ..., 1, 1, 1], shape=(2190,))

In [7]:
X_scaled = np.zeros_like(X_raw)
X_scaled

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(2190, 10))

In [8]:
X_means = np.mean(X_raw, axis=0)
X_stds = np.std(X_raw, axis=0)
X_stds[X_stds == 0] = 1e-8

In [9]:
for j in range(len(features)):
    X_scaled[:, j] = (X_raw[:, j] - X_means[j]) / X_stds[j]

print(X_scaled)

[[ 0.67170214 -0.91380916 -0.64219876 ... -0.72939738 -0.56090052
  -0.4652908 ]
 [ 1.04311572 -1.79828913 -1.35084574 ... -1.03280391 -0.6859253
   0.00962944]
 [ 1.85668833 -1.23222195 -1.50406671 ...  1.25653632 -0.43587575
  -0.37434863]
 ...
 [-0.05343865 -1.30298035 -1.46576147 ...  0.34631671 -0.81095007
   1.12114491]
 [ 1.53833383 -1.76290993 -1.6764403  ... -1.0052215  -0.81095007
  -0.38445331]
 [ 0.03499315 -0.91380916 -0.92948808 ... -0.75697979 -0.43587575
   2.6469525 ]]


## Sigmoid Function:

$g(z) = \frac{1}{1 + e^{-z}}$

$z = w^tx,\ \ \ $predicted data

In [10]:
def sigmoid(z):
    z = np.clip(z, -250, 250)
    return 1 / (1 + np.exp(-z))

## Prediction / Hypothesis:

$h_w(x) = w^tx$

In [11]:
def predict(x_row, w, b):
    z = b + np.dot(w, x_row)
    return sigmoid(z)

## Cost Function:
$J(w) = -\frac{1}{m}
\left[
\sum_{i=1}^{m} y_i \log\bigl(h_w(x_i)\bigr)
+
(1-y_i)\log\bigl(1-h_w(x_i)\bigr)
\right]$

In [12]:
def cost_function(X, Y, w, b, lambda_reg):
    m = len(Y)
    current_cost = 0.0
    epsilon = 1e-15

    for i in range(m):
        h_wx = predict(X[i], w, b)
        h_wx = max(epsilon, min(1 - epsilon, h_wx))

        current_cost += -(Y[i] * np.log(h_wx) + (1 - Y[i]) * np.log(1 - h_wx))
        
    # Calculate the loss
    loss = current_cost / m
    
    # Calculate the L2 Penalty
    l2_penalty = (lambda_reg / (2 * m)) * sum([w_j**2 for w_j in w])
    
    return loss + l2_penalty

## Parameter Update


In [13]:
def update_weights(X, Y, w, b, learning_rate, lambda_reg):
    m = len(Y)
    n = len(w)

    w_deriv = [0.0] * n
    b_deriv = 0.0
    
    for i in range(m):
        error = predict(X[i], w, b) - Y[i]
        b_deriv += error
        for j in range(n):
            w_deriv[j] += error * X[i][j]

    b -= learning_rate * (b_deriv / m)
    
    # L2 penalty directly into the weight update
    for j in range(n):
        regularized_deriv = (w_deriv[j] / m) - ((lambda_reg / m) * w[j])
        w[j] -= learning_rate * regularized_deriv

    return w, b

## Training Loop

In [14]:
m = len(Y)
n = len(features)

w = [0.0] * n 
b = 0.0

learning_rate = 0.1
epochs = 5000
tolerance = 0.0001
lambda_reg = 1.0

print("Starting Regularized Logistic Regression Training...\n")
prev_cost = cost_function(X_scaled, Y, w, b, lambda_reg)

for epoch in range(1, epochs + 1):
    w, b = update_weights(X_scaled, Y, w, b, learning_rate, lambda_reg)
    current_cost = cost_function(X_scaled, Y, w, b, lambda_reg)

    if epoch % 50 == 0 or epoch == 1:
        print(f"Epoch {epoch:4d}: Cost = {current_cost:.5f}")

    cost_decrease = prev_cost - current_cost
    if cost_decrease < tolerance:
        print(f"\n[Convergence Reached] Stopped early at Epoch {epoch}!")
        break

    prev_cost = current_cost

print("\nTraining Complete!")
print(f"Final Bias (b) : {b:.5f}")
for j in range(n):
    print(f"Weight {features[j]:15} : {w[j]:.5f}")

Starting Regularized Logistic Regression Training...

Epoch    1: Cost = 0.66956
Epoch   50: Cost = 0.38251
Epoch  100: Cost = 0.34816
Epoch  150: Cost = 0.33818

[Convergence Reached] Stopped early at Epoch 158!

Training Complete!
Final Bias (b) : 1.33230
Weight pressure        : -0.12985
Weight maxtemp         : 0.00661
Weight temparature     : 0.02881
Weight mintemp         : 0.04727
Weight dewpoint        : 0.23111
Weight humidity        : 0.37536
Weight cloud           : 0.88846
Weight sunshine        : -0.59221
Weight winddirection   : 0.02770
Weight windspeed       : 0.10259


In [15]:
# To predict for a specific day (already scaled):
test_day = X_scaled[4]

probability_of_rain = predict(test_day, w, b)

print(f"Probability of Rain: {probability_of_rain * 100:.1f}%")

if probability_of_rain >= 0.5:
    print("Prediction: It will RAIN (Class 1)")
else:
    print("Prediction: It will NOT rain (Class 0)")

Probability of Rain: 8.6%
Prediction: It will NOT rain (Class 0)


## Run on test data and find accuracy

In [16]:
test_data = pd.read_csv('test.csv') 

# NaN value removal with mean
for feature in features:
    test_data[feature] = test_data[feature].fillna(test_data[feature].mean())

In [17]:
X_test_raw = test_data[features].values
X_test_raw

array([[1.0195e+03, 1.7500e+01, 1.5800e+01, ..., 0.0000e+00, 5.0000e+01,
        2.4300e+01],
       [1.0165e+03, 1.7500e+01, 1.6500e+01, ..., 0.0000e+00, 5.0000e+01,
        3.5300e+01],
       [1.0239e+03, 1.1200e+01, 1.0400e+01, ..., 0.0000e+00, 4.0000e+01,
        1.6900e+01],
       ...,
       [1.0227e+03, 2.1000e+01, 1.8500e+01, ..., 0.0000e+00, 5.0000e+01,
        2.1900e+01],
       [1.0144e+03, 2.1000e+01, 2.0000e+01, ..., 0.0000e+00, 5.0000e+01,
        3.9500e+01],
       [1.0209e+03, 2.2200e+01, 1.8800e+01, ..., 2.0000e-01, 6.0000e+01,
        5.0600e+01]], shape=(730, 10))

In [18]:
# feature scaling in testing data
X_test_scaled = np.zeros_like(X_test_raw)
for j in range(len(features)):
    X_test_scaled[:, j] = (X_test_raw[:, j] - X_means[j]) / X_stds[j]

X_test_scaled

array([[ 1.04311572, -1.56832434, -1.56152458, ..., -1.03280391,
        -0.6859253 ,  0.25214191],
       [ 0.5125249 , -1.56832434, -1.42745623, ..., -1.03280391,
        -0.6859253 ,  1.36365737],
       [ 1.8213156 , -2.6827691 , -2.59576612, ..., -1.03280391,
        -0.81095007, -0.49560486],
       ...,
       [ 1.60907927, -0.94918836, -1.0444038 , ..., -1.03280391,
        -0.6859253 ,  0.00962944],
       [ 0.14111132, -0.94918836, -0.75711449, ..., -1.03280391,
        -0.6859253 ,  1.78805418],
       [ 1.29072478, -0.73691317, -0.98694594, ..., -0.97763909,
        -0.56090052,  2.90967433]], shape=(730, 10))

In [19]:
# Make Predictions
predictions = []
threshold = 0.5

for i in range(len(X_test_scaled)):
    prob = predict(X_test_scaled[i], w, b)
    
    if prob >= threshold:
        predictions.append(1)
    else:
        predictions.append(0)

predictions = np.array(predictions)

In [20]:
# Submission File
submission = pd.DataFrame({
    'id': test_data['id'],
    'rainfall': predictions
})

# Save it to a CSV file
submission.to_csv('submission.csv', index=False)

# Display the first 10 rows to verify it looks correct
print("\nPreview of Submission:")
print(submission.head(10))


Preview of Submission:
     id  rainfall
0  2190         1
1  2191         1
2  2192         1
3  2193         0
4  2194         0
5  2195         1
6  2196         1
7  2197         1
8  2198         1
9  2199         1


## Scikit-learn built-in model of Linear Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [22]:
log_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)

In [23]:

# Train the Model
print("Training Scikit-Learn Logistic Regression...")
log_model.fit(X_scaled, Y)

Training Scikit-Learn Logistic Regression...


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [24]:
# Scikit-Learn stores the bias in .intercept_ and the feature weights in .coef_
b_builtin = log_model.intercept_[0]
w_builtin = log_model.coef_[0]

print("Training Complete!\n")
print("-" * 50)
print(f"Built-in Bias (b) : {b_builtin:.4f}")
for j in range(len(features)):
    print(f"Built-in Weight {features[j]:15} : {w_builtin[j]:.4f}")

Training Complete!

--------------------------------------------------
Built-in Bias (b) : 1.6480
Built-in Weight pressure        : -0.1884
Built-in Weight maxtemp         : 0.1074
Built-in Weight temparature     : -0.1655
Built-in Weight mintemp         : -0.3034
Built-in Weight dewpoint        : 0.7496
Built-in Weight humidity        : 0.3189
Built-in Weight cloud           : 1.1401
Built-in Weight sunshine        : -0.5615
Built-in Weight winddirection   : -0.0279
Built-in Weight windspeed       : 0.1480


In [25]:
sklearn_predictions = log_model.predict(X_test_scaled)

print("Direct Class Predictions (First 5):")
print(sklearn_predictions[:5])

Direct Class Predictions (First 5):
[1 1 1 0 0]


# Comparing Manual model with Scikit-learn model

In [26]:
# Manual Prediction
manual_predictions = []
for i in range(len(X_test_scaled)):
    prob = predict(X_test_scaled[i], w, b)
    manual_predictions.append(1 if prob >= 0.5 else 0)

manual_predictions = np.array(manual_predictions)

# Scikit-learn prediction
sklearn_predictions = log_model.predict(X_test_scaled)

# How many times did both models guess the exact same thing?
agreements = np.sum(manual_predictions == sklearn_predictions)
total_samples = len(X_test_scaled)
agreement_rate = (agreements / total_samples) * 100

print(f"Total Test Samples : {total_samples}")
print(f"Models Agreed On   : {agreements}")
print(f"Agreement Rate     : {agreement_rate:.2f}%\n")

# --- 4. Visual Comparison ---
# Let's put them side-by-side in a DataFrame to see where they match or differ
comparison_df = pd.DataFrame({
    'id': test_data['id'],
    'Manual_Prediction': manual_predictions,
    'Sklearn_Prediction': sklearn_predictions,
    'Match': manual_predictions == sklearn_predictions
})

print("--- First 15 Predictions Head-to-Head ---")
print(comparison_df.head(15))

Total Test Samples : 730
Models Agreed On   : 716
Agreement Rate     : 98.08%

--- First 15 Predictions Head-to-Head ---
      id  Manual_Prediction  Sklearn_Prediction  Match
0   2190                  1                   1   True
1   2191                  1                   1   True
2   2192                  1                   1   True
3   2193                  0                   0   True
4   2194                  0                   0   True
5   2195                  1                   1   True
6   2196                  1                   1   True
7   2197                  1                   1   True
8   2198                  1                   1   True
9   2199                  1                   1   True
10  2200                  1                   1   True
11  2201                  0                   0   True
12  2202                  1                   1   True
13  2203                  1                   1   True
14  2204                  0                   0   True